# Chapter 39: Stereo Vision & Depth Estimation
## Multimedia & Computer Vision Learning Hub - ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Objective:** Mempelajari geometri epipolar, matriks esensial & fundamental, triangulasi disparitas, serta analisis ketidakpastian kedalaman metrik.

### 1. Landasan Teori & Formulasi Matematis
Kendala epipolar dua pandangan:
$$\mathbf{x}_R^T \mathbf{F} \mathbf{x}_L = 0, \quad \text{di mana } \mathbf{F} = \mathbf{K}_R^{-T} [\mathbf{t}]_\times \mathbf{R} \mathbf{K}_L^{-1}$$

Persamaan kedalaman stereo fundamental hasil triangulasi:
$$Z = \frac{f \cdot B}{d}, \quad d = u_L - u_R$$

Pertumbuhan galat kuadratik ketidakpastian kedalaman:
$$|\Delta Z| \approx \frac{Z^2}{f \cdot B} |\Delta d|$$

In [ ]:
# 2. Dependency Setup & Environment
import numpy as np
import cv2
print(f"NumPy Version: {np.__version__}")
print(f"OpenCV Version: {cv2.__version__}")

In [ ]:
# 3. Sintesis Pasangan Citra Stereo Terektifikasi (Stereo Left-Right Pair)
# Buat pemandangan sintetis dengan latar belakang (jauh) dan kotak latar depan (dekat)
H, W = 240, 320
img_left = np.zeros((H, W), dtype=np.uint8)
img_right = np.zeros((H, W), dtype=np.uint8)

# Latar belakang bertekstur grid (disparitas kecil d=4 px -> jarak jauh)
img_left[::16, :] = 100
img_left[:, ::16] = 100
# Geser 4 piksel ke kiri untuk citra kanan
img_right[:, :-4] = img_left[:, 4:]

# Kotak latar depan (disparitas besar d=24 px -> jarak dekat)
# Kotak pada citra kiri di (80:160, 100:180)
img_left[80:160, 100:180] = 220
# Pada citra kanan, kotak bergeser ke kiri sebesar disparitas 24 piksel (100-24 = 76)
img_right[80:160, 76:156] = 220

print(f"Citra Stereo berhasil dibuat: Left {img_left.shape}, Right {img_right.shape}")

In [ ]:
# 4. Eksekusi Algoritma Stereo Block Matching (StereoBM)
# Inisialisasi StereoBM dengan 32 tingkat disparitas dan ukuran blok 15x15
stereo = cv2.StereoBM_create(numDisparities=32, blockSize=15)
disparity = stereo.compute(img_left, img_right)

# Nilai disparitas keluaran StereoBM disimpan dalam skala fixed-point 16-bit (dikalikan 16)
disp_float = disparity.astype(np.float32) / 16.0

print(f"Rentang Nilai Disparitas Terestimasi: [{disp_float.min()}, {disp_float.max()}] piksel")
# Nilai disparitas rata-rata pada area kotak latar depan
foreground_disp = disp_float[90:150, 110:170]
valid_disp = foreground_disp[foreground_disp > 0]
print(f"Disparitas Latar Depan Terdeteksi: {np.median(valid_disp):.1f} px (Ground Truth = 24.0 px)")

In [ ]:
# 5. Rekonstruksi Peta Kedalaman Metrik & Analisis Sensitivitas Kuadratik
focal_length = 500.0  # px
baseline = 0.25       # meter (25 cm)

# Hindari pembagian nol pada disparitas non-positif
depth_map = np.where(disp_float > 0, (focal_length * baseline) / disp_float, 0.0)

near_depth = (focal_length * baseline) / 24.0
far_depth = (focal_length * baseline) / 4.0
print(f"Jarak Objek Dekat (d=24 px): {near_depth:.2f} meter")
print(f"Jarak Latar Belakang (d=4 px): {far_depth:.2f} meter")

# Hitung rasio ketidakpastian kedalaman untuk kesalahan 1 piksel disparitas
err_near = (near_depth ** 2) / (focal_length * baseline)
err_far = (far_depth ** 2) / (focal_length * baseline)
print(f"Ketidakpastian pada 5.2m: +- {err_near*100:.2f} cm")
print(f"Ketidakpastian pada 31.25m: +- {err_far:.2f} meter ({err_far/err_near:.1f}x lebih besar!)")
assert err_far > err_near * 30, "Ketidakpastian jarak jauh harus melonjak secara kuadratik"

### 6. Batasan & Analisis Kegagalan (Failure Modes)
1. **Area Oklusi (Dead Zones)**: Bagian yang hanya tampak oleh satu kamera menghasilkan nilai disparitas invalid (-1 atau 0).
2. **Pantulan & Pola Repetitif**: Permukaan kaca, genangan air, atau kisi-kisi seragam memicu kecocokan salah pada scanline.
3. **Penyimpangan Sumbu Optik Vertikal**: Jika kamera tidak terektifikasi sempurna (terdapat rotasi pitch atau roll relatif), algoritma scanline 1D gagal menemukan pasangan.

### 7. Referensi Akademik
- Scharstein, D., & Szeliski, R. (2002). *A Taxonomy and Evaluation of Two-Frame Stereo Correspondence Algorithms*. IJCV.
- Hirschmüller, H. (2008). *Stereo Processing by Semiglobal Matching and Mutual Information*. IEEE TPAMI.